<a href="https://colab.research.google.com/github/SeoungmoY/pain_free/blob/main/pubmed_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# 0. 코랩에 번역기 기능을 설치합니다. (처음 한 번만 설치됨)
!pip install deep-translator -q

import requests
import xml.etree.ElementTree as ET
import time
import pandas as pd
from deep_translator import GoogleTranslator

# ==========================================
# 1. 자연어 검색어 입력 및 자동 번역 (AI 마법 🧙‍♂️)
# ==========================================
# 코드를 실행하면 아래의 질문이 화면에 나타나고 입력을 기다립니다.
korean_input = input("💡 찾고 싶은 의학 주제를 일상적인 말로 편하게 적어주세요.\n(예: 종양 환자의 영양 및 면역 상태 평가를 위한 HALP 점수 관련 최근 연구 찾아줘)\n입력창 ✏️: ")

print("\n🔄 입력하신 문장을 PubMed가 이해할 수 있는 영어로 번역 중입니다...")
# 구글 번역기를 이용해 한국어 -> 영어로 자동 변환
english_query = GoogleTranslator(source='ko', target='en').translate(korean_input)
print(f"👉 번역된 검색어: {english_query}\n")

# ==========================================
# 2. 기본 설정 및 고급 필터 장착
# ==========================================
api_key = "5614f13bc4b209367f9178c35f2b0040fc08"
max_results = 5

# 조건: RCT or Systematic Review + 무료 원문 + 최근 10년
advanced_filter = ' AND (Randomized Controlled Trial[ptyp] OR Systematic Review[ptyp]) AND "loattrfree full text"[sb] AND "last 10 years"[dp]'

# 번역된 자연어 문장 뒤에 고급 필터를 꼬리표처럼 붙여줍니다.
final_search_term = f"({english_query}){advanced_filter}"

excel_data_list = []

# ==========================================
# 3. 논문 검색 (ESearch)
# ==========================================
search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
search_params = {
    "db": "pubmed",
    "term": final_search_term,
    "retmax": max_results,
    "retmode": "json",
    "api_key": api_key
}

search_response = requests.get(search_url, params=search_params).json()
pmid_list = search_response.get('esearchresult', {}).get('idlist', [])

if not pmid_list:
    print("❌ 조건에 딱 맞는 논문이 없습니다. 문장을 조금 다르게 입력해 보세요.")
else:
    pmid_string = ",".join(pmid_list)
    print(f"✅ {len(pmid_list)}개의 최상위 근거 논문을 찾았습니다. 원문과 데이터를 가져옵니다...\n")

    # ==========================================
    # 4. 논문 상세 정보 수집 (EFetch) 및 PDF 다운로드
    # ==========================================
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {
        "db": "pubmed",
        "id": pmid_string,
        "retmode": "xml",
        "api_key": api_key
    }

    fetch_response = requests.get(fetch_url, params=fetch_params)
    root = ET.fromstring(fetch_response.content)

    print("=" * 70)

    for article in root.findall('.//PubmedArticle'):
        title_element = article.find('.//ArticleTitle')
        title = title_element.text if title_element is not None else "제목 없음"

        pmid = article.find('.//PMID').text
        pmcid = None

        for article_id in article.findall('.//ArticleId'):
            if article_id.get('IdType') == 'pmc':
                pmcid = article_id.text
                break

        abstract_text = ""
        for abs_elem in article.findall('.//AbstractText'):
            if abs_elem.text:
                abstract_text += abs_elem.text + " "

        pubmed_url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

        excel_data_list.append({
            "논문 제목": title,
            "PMID": pmid,
            "원문 링크": pubmed_url,
            "초록 (Abstract)": abstract_text.strip()
        })

        print(f"📌 제목: {title}")
        print(f"🔗 논문 링크: {pubmed_url}")

        if pmcid:
            pdf_url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/pdf/"
            headers = {'User-Agent': 'Mozilla/5.0'}
            try:
                pdf_response = requests.get(pdf_url, headers=headers)
                if pdf_response.status_code == 200 and b"%PDF" in pdf_response.content[:5]:
                    file_name = f"{pmcid}_{pmid}.pdf"
                    with open(file_name, "wb") as file:
                        file.write(pdf_response.content)
                    print(f"💾 [PDF 저장 완료] {file_name}")
                else:
                    print("⚠️ [주의] 출판사 홈페이지로 이동해야만 다운로드 가능한 형식입니다.")
            except Exception as e:
                print(f"❌ 다운로드 중 에러 발생: {e}")
        else:
            print("🔒 [알림] PMCID를 찾을 수 없습니다.")

        print("=" * 70)
        time.sleep(1)

# ==========================================
# 5. 엑셀 파일 생성
# ==========================================
if excel_data_list:
    df = pd.DataFrame(excel_data_list)
    excel_filename = "PubMed_AI_Search_Results.xlsx"
    df.to_excel(excel_filename, index=False)
    print(f"\n🎉 모든 작업 완료! 엑셀 파일 [{excel_filename}]이 성공적으로 생성되었습니다!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.2 MB/s eta 0:00:00
💡 찾고 싶은 의학 주제를 일상적인 말로 편하게 적어주세요.
(예: 종양 환자의 영양 및 면역 상태 평가를 위한 HALP 점수 관련 최근 연구 찾아줘)
입력창 ✏️: 위암의 전이과정

🔄 입력하신 문장을 PubMed가 이해할 수 있는 영어로 번역 중입니다...
👉 번역된 검색어: Metastatic process of stomach cancer

✅ 5개의 최상위 근거 논문을 찾았습니다. 원문과 데이터를 가져옵니다...

📌 제목: Prevalence of FGFR2b Protein Overexpression in Advanced Gastric Cancers During Prescreening for the Phase III FORTITUDE-101 Trial.
🔗 논문 링크: https://pubmed.ncbi.nlm.nih.gov/39854659/
⚠️ [주의] 출판사 홈페이지로 이동해야만 다운로드 가능한 형식입니다.
📌 제목: None
🔗 논문 링크: https://pubmed.ncbi.nlm.nih.gov/31542733/
⚠️ [주의] 출판사 홈페이지로 이동해야만 다운로드 가능한 형식입니다.
📌 제목: Cryotherapy for liver metastases.
🔗 논문 링크: https://pubmed.ncbi.nlm.nih.gov/31291464/
⚠️ [주의] 출판사 홈페이지로 이동해야만 다운로드 가능한 형식입니다.
📌 제목: Systematic vs. on-demand early palliative care in gastric cancer patients: a randomized clinical trial assessing patient and healthcare service outcomes.
🔗 논문 링크: https://pubmed.ncbi.nlm.nih.gov/30357555/
🔒 [알림] PMCID